# 07 — Spark ML: Pipelines and Feature Engineering

The `pyspark.ml` Transformer/Estimator/Pipeline abstractions, and the feature-engineering building blocks (`StringIndexer`, `OneHotEncoder`, `VectorAssembler`, `StandardScaler`) every Spark ML workflow is built from. This is the part of Spark ML a **data engineer** interview actually probes — building correct, leak-free, production feature pipelines at scale, not model tuning.

> **Setup note:** these notebooks are written but **not executed** — PySpark is not
> installed in this environment. To run them locally:
>
> ```bash
> python -m venv .venv && source .venv/bin/activate
> pip install pyspark==3.5.1
> # Java 11/17 must be on PATH (java -version)
> jupyter notebook
> ```
>
> Everything below is correct, runnable PySpark MLlib (`pyspark.ml`) — read it as
> a reference and run cell-by-cell once your environment is set up.

## 0. `pyspark.ml` vs `pyspark.mllib` — say this early if asked

- **`pyspark.mllib`** — the original RDD-based API. **Deprecated**, maintenance mode only. If you see it in a codebase, it's legacy.
- **`pyspark.ml`** — the current DataFrame-based API. Everything below uses this. It integrates with the same DataFrames/Catalyst optimizer used elsewhere in Spark, and is the only one still developed.

**Interview one-liner:** "Always `pyspark.ml`, never `pyspark.mllib` — the latter is deprecated."

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("spark-ml-pipelines")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

customers = spark.createDataFrame(
    [
        (1, "month-to-month", "fiber", 29.99, 2, 0),
        (2, "one-year",       "dsl",   19.50, 24, 0),
        (3, "month-to-month", "fiber", 89.10, 1, 1),
        (4, "two-year",       "dsl",   15.00, 36, 0),
        (5, "month-to-month", "fiber", 95.40, 3, 1),
        (6, "one-year",       "none",  10.00, 18, 0),
    ],
    ["customer_id", "contract_type", "internet_service", "monthly_charges", "tenure_months", "churned"],
)
customers.show()

## 1. The three core abstractions

- **Transformer** — has a `.transform(df)` method; stateless, no learning involved (or already-learned). Examples: a fitted `StringIndexerModel`, `VectorAssembler`, `Tokenizer`.
- **Estimator** — has a `.fit(df)` method; *learns* parameters from data and returns a **Transformer**. Examples: `StringIndexer` (learns the label→index mapping), `StandardScaler` (learns mean/stddev), `LogisticRegression` (learns coefficients).
- **Pipeline** — itself an Estimator: it chains a list of stages (Transformers and/or Estimators). `Pipeline(stages=[...]).fit(train)` runs each stage's `.fit()`/`.transform()` in order and returns a **`PipelineModel`** (a Transformer) that you call `.transform(test)` on.

**Why this abstraction matters for a data engineer:** fitting each stage's parameters (e.g. StandardScaler's mean/stddev, StringIndexer's category mapping) **only on the training set** and then just calling `.transform()` on the test/production set — never re-fitting — is what prevents data leakage. `Pipeline` makes that the natural way to write the code instead of something you have to remember to do by hand.

## 2. Encoding categoricals: `StringIndexer` + `OneHotEncoder`

MLlib algorithms need numeric input — categorical string columns must be converted first.

- **`StringIndexer`** — maps each distinct string to a numeric index, ordered by frequency (`0` = most frequent category). This alone is **not** appropriate to feed a linear model directly — it implies a false ordinal relationship (category `2` isn't "twice" category `1`).
- **`OneHotEncoder`** — takes the indexed column and expands it into a sparse vector (one "hot" position per category), removing the false ordinality. Tree-based models (Random Forest, GBT) can often skip this and use the raw indexed column directly, since trees split on thresholds rather than assuming linear order — but linear/logistic regression should always one-hot encode.

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder

contract_indexer = StringIndexer(inputCol="contract_type", outputCol="contract_idx")
service_indexer = StringIndexer(inputCol="internet_service", outputCol="service_idx")

# fit + transform shown standalone here for illustration; inside a real
# Pipeline you'd never call these individually — the Pipeline does it for you
indexed = contract_indexer.fit(customers).transform(customers)
indexed = service_indexer.fit(indexed).transform(indexed)
indexed.select("contract_type", "contract_idx", "internet_service", "service_idx").show()

encoder = OneHotEncoder(inputCols=["contract_idx", "service_idx"],
                        outputCols=["contract_vec", "service_vec"])
encoded = encoder.fit(indexed).transform(indexed)
encoded.select("contract_vec", "service_vec").show(truncate=False)

## 3. Assembling features: `VectorAssembler`

Every MLlib estimator expects **one column** containing a `Vector` (named `features` by convention) — you can't pass several separate numeric columns directly. `VectorAssembler` combines multiple numeric and vector columns into that single `features` column. This is the step every Spark ML pipeline has, and the one most often forgotten by someone new to the API.

In [ ]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["contract_vec", "service_vec", "monthly_charges", "tenure_months"],
    outputCol="features",
)
assembled = assembler.transform(encoded)
assembled.select("features", "churned").show(truncate=False)

## 4. Scaling: `StandardScaler`

Distance-/gradient-based algorithms (logistic regression, SVM, k-means) are sensitive to feature scale — `monthly_charges` (range ~10-100) would dominate `tenure_months` (range ~1-36) purely because of units, not actual importance. `StandardScaler` (zero mean, unit variance) — or `MinMaxScaler` — is fit **only on training data** and applied identically to test data via the same `Pipeline`. Tree-based models (Random Forest, GBT) are scale-invariant and don't need this step.

In [ ]:
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(inputCol="features", outputCol="scaled_features",
                        withMean=True, withStd=True)
scaled = scaler.fit(assembled).transform(assembled)
scaled.select("scaled_features").show(truncate=False)

## 5. Putting it together: one `Pipeline`

Rewriting all four steps above as pipeline stages — this is how it actually gets written in practice. Note stages are declared **unfit** (`StringIndexer(...)`, not `StringIndexer(...).fit(df)`) — the `Pipeline` calls `.fit()` on each Estimator stage itself, in order, threading the output columns forward.

In [ ]:
from pyspark.ml import Pipeline

train, test = customers.randomSplit([0.8, 0.2], seed=42)

feature_pipeline = Pipeline(stages=[
    StringIndexer(inputCol="contract_type", outputCol="contract_idx"),
    StringIndexer(inputCol="internet_service", outputCol="service_idx"),
    OneHotEncoder(inputCols=["contract_idx", "service_idx"],
                  outputCols=["contract_vec", "service_vec"]),
    VectorAssembler(inputCols=["contract_vec", "service_vec", "monthly_charges", "tenure_months"],
                    outputCol="raw_features"),
    StandardScaler(inputCol="raw_features", outputCol="features"),
])

fitted_pipeline = feature_pipeline.fit(train)   # learns indices + scaler stats from TRAIN only
train_ready = fitted_pipeline.transform(train)
test_ready = fitted_pipeline.transform(test)     # re-uses train's fitted mapping/stats — no leakage
test_ready.select("features", "churned").show(truncate=False)

## 6. Interview Q&A

1. **"Why can't you just pass raw columns into `LogisticRegression`?"** — every `pyspark.ml` estimator expects a single `features` column of type `Vector`; `VectorAssembler` builds that column from your raw features.
2. **"What's the data-leakage risk in feature engineering, and how does `Pipeline` prevent it?"** — fitting a scaler/indexer on the full dataset (train+test) lets test-set statistics influence the transformation applied to training data. `Pipeline.fit(train)` learns parameters from train only; `pipelineModel.transform(test)` reuses those exact learned parameters — never refit on test.
3. **"When would you skip one-hot encoding and just use `StringIndexer`'s raw index?"** — for tree-based models (Random Forest, GBT, Decision Trees), which split on thresholds and don't assume linear ordering, so the false ordinal relationship doesn't hurt them the way it would a linear model.
4. **"Is `VectorAssembler` an Estimator or a Transformer?"** — a Transformer — it has no parameters to learn from data, it just concatenates columns, so it only implements `.transform()`.

## Summary

- Transformer = `.transform()` only (stateless); Estimator = `.fit()` → produces a Transformer; `Pipeline` chains both and is itself an Estimator.
- Feature pipeline shape: `StringIndexer` → `OneHotEncoder` → `VectorAssembler` → (optional) `StandardScaler`.
- Fit the whole pipeline on train only; call `.transform()` on test — this is what prevents leakage.
- Next: `08_spark_ml_classification_regression_and_evaluation.ipynb`.